# Laboratorio 6 -- Análisis de redes sociales en YouTube

Universidad del Valle de Guatemala. CC3084 Data Science, Semestre II 2026.

Fernando Rueda -- 23748. Fernando Hernández -- 23645.

Estudiamos la estructura de participación en YouTube a partir de dos conjuntos,
uno de videos y canales y otro de comentarios. Este avance cubre la carga y la
integración de los datos, un diagnóstico de calidad, la limpieza y el
preprocesamiento del texto, el análisis exploratorio y la construcción de la red
bipartita autor-video. Los datos no permiten saber quién respondió a quién, así
que el conteo de respuestas no se interpreta como una relación entre usuarios.

## Ejercicio 1. Carga, comprensión e integración de los datos

### 1.1 Carga de los archivos

Cargamos los dos archivos con pandas y revisamos sus dimensiones.

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SALIDAS = ROOT / "salidas"
SALIDAS.mkdir(exist_ok=True)

videos = pd.read_csv(ROOT / "data" / "youtube_videos.csv")
comentarios = pd.read_csv(ROOT / "data" / "youtube_comments.csv")
print("videos:", videos.shape, "| comentarios:", comentarios.shape)

videos: (293, 20) | comentarios: (406, 17)


### 1.2 Unidad de observación, llave primaria y variables relevantes

En el archivo de videos cada fila es un video de YouTube, la llave primaria es
video_id y las variables más relevantes son el canal que lo publicó, con channel_id
como identificador estable y channel_name como nombre visible, el título, la
categoría, el número de visualizaciones y la consulta con la que se recolectó. En
el archivo de comentarios cada fila es un comentario principal, la llave primaria es
comment_id y las variables clave son el texto del comentario, el autor, con
author_channel_id como identificador y author_name como nombre visible, el video al
que pertenece a través de video_id, y los conteos de me gusta y de respuestas.

In [2]:
def resumen(df, nombre):
    print(f"== {nombre} ==")
    print("dimensiones:", df.shape)
    print("columnas:", list(df.columns))

resumen(videos, "videos")
print()
resumen(comentarios, "comentarios")
print("\n¿video_id es único en videos?", videos["video_id"].is_unique)
print("¿comment_id es único en comentarios?", comentarios["comment_id"].is_unique)

== videos ==
dimensiones: (293, 20)
columnas: ['video_id', 'title', 'channel_name', 'channel_id', 'source_query', 'source_group', 'dataset_sources', 'channel_handle', 'published_time', 'view_count_text', 'description_snippet', 'video_url', 'query_hits', 'keywords', 'description', 'view_count', 'publish_date', 'upload_date', 'category', 'owner_handle']

== comentarios ==
dimensiones: (406, 17)
columnas: ['video_id', 'comment_id', 'video_title', 'channel_name', 'channel_id', 'author_name', 'author_channel_id', 'text', 'source_query', 'source_group', 'dataset_sources', 'author_handle', 'published_text', 'like_count_text', 'reply_count', 'is_pinned', 'viewer_rating']

¿video_id es único en videos? True
¿comment_id es único en comentarios? True


### 1.3 Relación entre canal, video, autor, comentario, categoría y consulta

Un canal, identificado por channel_id, publica videos, y cada video pertenece a una
categoría de YouTube y fue recolectado mediante una consulta de búsqueda descrita por
source_query y source_group. Cada comentario se publica en un video, relación que se
establece con video_id, y lo escribe un autor identificado por author_channel_id. Es
importante notar que el canal del video y el autor del comentario son entidades
distintas, el primero es quien subió el video y el segundo quien comentó, y que un
mismo autor puede comentar en varios videos. El conteo de respuestas de un comentario
no dice quién respondió, así que no define una relación entre autores.

### 1.4 Integración por video_id

Unimos los comentarios con la información de su video a través de video_id y
reportamos cuántos comentarios pudieron asociarse a un video del conjunto.

In [3]:
ids_video = set(videos["video_id"])
con_video = comentarios["video_id"].isin(ids_video)
print("comentarios totales:", len(comentarios))
print("comentarios asociados a un video del conjunto:", int(con_video.sum()))
print("comentarios sin video en el conjunto:", int((~con_video).sum()))
print("videos con al menos un comentario:", comentarios["video_id"].nunique())

integrado = comentarios.merge(
    videos, on="video_id", how="left", suffixes=("_com", "_vid"))
print("\ntabla integrada:", integrado.shape)

comentarios totales: 406
comentarios asociados a un video del conjunto: 406
comentarios sin video en el conjunto: 0
videos con al menos un comentario: 19

tabla integrada: (406, 36)


## Ejercicio 2. Calidad, limpieza y preprocesamiento

### 2.1 Diagnóstico inicial de calidad

Revisamos dimensiones, tipos, valores faltantes, duplicados, variables constantes y
la consistencia de los identificadores en los dos conjuntos.

In [4]:
def diagnostico(df, nombre):
    print(f"===== {nombre} =====")
    print("dimensiones:", df.shape)
    print("filas duplicadas:", int(df.duplicated().sum()))
    constantes = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    print("variables constantes:", constantes)
    faltantes = df.isna().mean().mul(100).round(1)
    faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
    print("variables con faltantes (%):")
    print(faltantes.to_string() if len(faltantes) else "  ninguna")
    print()

diagnostico(videos, "videos")
diagnostico(comentarios, "comentarios")

===== videos =====
dimensiones: (293, 20)
filas duplicadas: 0
variables constantes: []
variables con faltantes (%):
description            8.9
description_snippet    8.5
published_time         4.4
view_count_text        4.4

===== comentarios =====
dimensiones: (406, 17)
filas duplicadas: 0
variables constantes: ['is_pinned', 'viewer_rating']
variables con faltantes (%):
viewer_rating    100.0



In [5]:
# consistencia entre identificador de canal y nombre visible
print("canales por channel_id:", videos["channel_id"].nunique())
print("nombres de canal distintos:", videos["channel_name"].nunique())
dup_nombre = (videos.groupby("channel_name")["channel_id"].nunique() > 1).sum()
print("nombres de canal usados por más de un channel_id:", int(dup_nombre))
print("\nautores por author_channel_id:", comentarios["author_channel_id"].nunique())
print("nombres de autor distintos:", comentarios["author_name"].nunique())

canales por channel_id: 97
nombres de canal distintos: 97
nombres de canal usados por más de un channel_id: 0

autores por author_channel_id: 332
nombres de autor distintos: 332


### 2.2 Variables problemáticas y su tratamiento

Varias variables requieren cuidado. La columna is_pinned vale False en todos los
comentarios y viewer_rating está vacía en los 406 registros, así que ninguna aporta
información y las descartamos. Las variables de tiempo published_time y published_text
son relativas, del tipo hace dos días, y dependen del momento de recolección, así que
no las convertimos a fecha exacta y solo las conservamos como referencia. Los conteos
view_count_text y like_count_text vienen como texto con comas, con la palabra vistas o
con espacios en blanco, así que hay que convertirlos a número. Los nombres visibles de
canal y de autor pueden repetirse o cambiar, por lo que usamos los identificadores y no
los nombres para cualquier operación de llave.

### 2.3 Normalización de identificadores

Dejamos los identificadores como texto sin espacios sobrantes y conservamos los nombres
visibles aparte, sin sustituir los identificadores por ellos.

In [6]:
for df, cols in [(videos, ["video_id", "channel_id"]),
                (comentarios, ["video_id", "comment_id", "channel_id", "author_channel_id"])]:
    for c in cols:
        df[c] = df[c].astype(str).str.strip()
print("identificadores normalizados como texto sin espacios")
print("ejemplo video_id:", videos["video_id"].head(2).tolist())
print("ejemplo author_channel_id:", comentarios["author_channel_id"].head(2).tolist())

identificadores normalizados como texto sin espacios
ejemplo video_id: ['-5puKGEqcUc', '-E7OPOLjMug']
ejemplo author_channel_id: ['UCdFlugHJJa4l3YqWuNRmvXw', 'UCvl1tzQeBeGy6efPTRJXSCw']


### 2.4 Conversión de conteos a numérico

Convertimos los conteos guardados como texto a número. Quitamos la palabra vistas, las
comas que separan miles y los espacios, e interpretamos los sufijos de mil y millón por
si aparecen. Los espacios en blanco de like_count_text corresponden a comentarios sin me
gusta, así que los tratamos como cero.

In [7]:
def a_numero(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower().strip()
    s = re.sub(r"vistas?|views?", "", s).replace(",", "").strip()
    if s == "":
        return np.nan
    mult = 1
    if s.endswith("k"):
        mult, s = 1_000, s[:-1]
    elif s.endswith("m"):
        mult, s = 1_000_000, s[:-1]
    try:
        return float(s) * mult
    except ValueError:
        return np.nan

videos["view_count_num"] = videos["view_count_text"].apply(a_numero)
comentarios["like_count"] = comentarios["like_count_text"].apply(a_numero).fillna(0).astype(int)
comentarios["reply_count"] = pd.to_numeric(comentarios["reply_count"], errors="coerce").fillna(0).astype(int)

# descartamos las variables sin información
comentarios = comentarios.drop(columns=["is_pinned", "viewer_rating"])
print("view_count_num (muestra):", videos["view_count_num"].dropna().head(4).tolist())
print("like_count (distribución):", comentarios["like_count"].describe()[["min", "mean", "max"]].round(2).to_dict())
print("columnas sin información eliminadas: is_pinned, viewer_rating")

view_count_num (muestra): [2390.0, 4.0, 29736.0, 383.0]
like_count (distribución): {'min': 0.0, 'mean': 5.73, 'max': 405.0}
columnas sin información eliminadas: is_pinned, viewer_rating
